# 13F Summary

Notebook version of python/13f_summary.py. This notebook will:
- Download the SEC EDGAR master index for Q1 2026 (or specified quarters),
- Find 13F-HR filings within the timeframe (2026-01-01 → 2026-03-09 by default),
- Parse each filing's INFORMATION TABLE XML to extract holdings,
- Select the top N filers by reported 13F assets (defaults to top 50),
- Aggregate holdings and compute net changes (latest − earliest),
- Exclude ETFs (configurable), apply a $1M minimum-change filter (configurable),
- Produce Top 10 holdings, Top 10 buys, Top 10 sells and save CSVs.


## Before running
- Edit the `USER_AGENT` variable in the next cell to include a valid contact per SEC policy, e.g.:
  `nicksirip-13f-summary/1.0 (email: youremail@example.com)`
- Install dependencies in your environment:
```
pip install requests lxml pandas python-dateutil
```
- This notebook will access sec.gov directly when run in VS Code (ensure internet access).


In [ ]:
# ---- CONFIG ----
import datetime as dt

# Edit this to include a real contact per SEC policy
USER_AGENT = "nicksirip-13f-summary-script/1.0 (youremail@example.com)"

START_DATE = dt.date(2026, 1, 1)
END_DATE = dt.date(2026, 3, 9)
QTR_MASTER_URL_TEMPLATE = "https://www.sec.gov/Archives/edgar/full-index/{year}/QTR{qtr}/master.idx"
MIN_CHANGE_USD = 1000000  # $1M threshold
TOP_FILERS_N = 50
INCLUDE_ETFS = false  # exclude ETFs by default
OUTPUT_PREFIX = "13f_summary_2026-Q1_2026-01-01_to_2026-03-09"
HEADERS = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate", "Host": "www.sec.gov"}
# ---- END CONFIG ----


In [ ]:
import re
import time
import logging
import requests
import pandas as pd
from dateutil import parser as dateparse
from lxml import etree
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

def fetch_master_index(year, qtr, headers=None):
    url = QTR_MASTER_URL_TEMPLATE.format(year=year, qtr=qtr)
    logging.info("Downloading master index: %s", url)
    r = requests.get(url, headers=headers or HEADERS, timeout=60)
    r.raise_for_status()
    return r.text


In [ ]:
def parse_master_idx_for_13f(master_idx_text, start_date, end_date):
    lines = master_idx_text.splitlines()
    start_i = 0
    for i, ln in enumerate(lines[:80]):
        if ln.strip().startswith("CIK|Company Name|Form Type|Date Filed|Filename"):
            start_i = i + 1
            break
    rows = []
    for ln in lines[start_i:]:
        if not ln.strip():
            continue
        parts = ln.split('|')
        if len(parts) < 5:
            continue
        cik, company_name, form_type, date_filed_str, filename = parts[:5]
        try:
            date_filed = dateparse.parse(date_filed_str).date()
        except Exception:
            continue
        if form_type.strip().upper() != "13F-HR":
            continue
        if not (start_date <= date_filed <= end_date):
            continue
        rows.append({
            "cik": cik.strip(),
            "company_name": company_name.strip(),
            "form_type": form_type.strip(),
            "date_filed": date_filed,
            "filename": filename.strip()
        })
    logging.info("Found %d 13F-HR filings in master index within date range", len(rows))
    return rows


In [ ]:
def fetch_filing_txt(filename, headers=None):
    base = "https://www.sec.gov/Archives/"
    url = base + filename
    logging.info("Fetching filing text: %s", url)
    r = requests.get(url, headers=headers or HEADERS, timeout=60)
    r.raise_for_status()
    return r.text


In [ ]:
def locate_information_table_xml_from_text(filing_text):
    m = re.search(r'(<informationTable[\s\S]*?</informationTable>)', filing_text, flags=re.IGNORECASE)
    if m:
        return m.group(1), None
    m2 = re.search(r'(<\?xml[\s\S]*?</informationTable>)', filing_text, flags=re.IGNORECASE)
    if m2:
        return m2.group(1), None
    xml_files = re.findall(r'([^\s,\"]+\.xml)', filing_text, flags=re.IGNORECASE)
    xml_files = [x for x in xml_files if 'infotable' in x.lower() or x.lower().endswith('.xml')]
    if xml_files:
        return None, xml_files[0]
    return None, None


In [ ]:
def fetch_xml_by_path(xml_path, headers=None):
    base = "https://www.sec.gov/Archives/"
    url = xml_path if xml_path.startswith("http") else base + xml_path
    logging.info("Fetching XML at: %s", url)
    r = requests.get(url, headers=headers or HEADERS, timeout=60)
    r.raise_for_status()
    return r.text


In [ ]:
def parse_information_table_xml(xml_text):
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(xml_text.encode('utf-8'), parser=parser)
    except Exception:
        try:
            wrapped = f"<root>{xml_text}</root>"
            root = etree.fromstring(wrapped.encode('utf-8'), parser=etree.XMLParser(recover=True))
        except Exception as e:
            logging.error("XML parsing failed: %s", e)
            return []
    info_tables = root.findall(".//{*}infoTable") or root.findall(".//infoTable") or root.findall(".//{*}informationTable")
    holdings = []
    if not info_tables:
        for elem in root.iter():
            if elem.tag.lower().endswith('infotable'):
                info_tables.append(elem)
    for info in info_tables:
        try:
            def find_text(parent, tag_name):
                el = parent.find(".//{*}" + tag_name)
                if el is None:
                    for ch in parent:
                        if ch.tag.lower().endswith(tag_name.lower()):
                            return (ch.text or "").strip()
                    return ""
                return (el.text or "").strip()
            name = find_text(info, "nameOfIssuer") or find_text(info, "nameofIssuer")
            title = find_text(info, "titleOfClass")
            cusip = find_text(info, "cusip")
            value_str = find_text(info, "value")
            value = 0
            try:
                value = int(re.sub(r'[^\d\-]', '', value_str)) * 1000
            except:
                try:
                    value = int(float(value_str) * 1000)
                except:
                    value = 0
            shrs_el = info.find(".//{*}shrsOrPrnAmt")
            shares = None
            if shrs_el is not None:
                amount = None
                for ch in shrs_el:
                    if ch.tag.lower().endswith("sshPrnamt") or ch.tag.lower().endswith("sshprnamt"):
                        amount = ch.text
                if amount is None:
                    for ch in shrs_el:
                        if ch.text and re.search(r'\d', ch.text):
                            amount = ch.text
                            break
                if amount:
                    try:
                        shares = float(re.sub(r'[^\d\.\-]', '', amount))
                    except:
                        shares = None
            holdings.append({
                "name": name,
                "title": title,
                "cusip": cusip,
                "value": value,
                "shares": shares
            })
        except Exception as ex:
            logging.exception("Failed parsing infoTable entry: %s", ex)
    return holdings


In [ ]:
def is_etf(holding):
    name = (holding.get("name") or "").lower()
    title = (holding.get("title") or "").lower()
    if "exchange traded fund" in name or "exchange traded fund" in title:
        return True
    if "etf" in name.split() or "etf" in title.split():
        return True
    return False


In [ ]:
def aggregate_holdings_by_security(holdings_list):
    agg = {}
    for h in holdings_list:
        key = h.get("cusip") or h.get("name")
        if not key:
            continue
        if key not in agg:
            agg[key] = {"name": h.get("name"), "cusip": h.get("cusip"), "value": 0, "shares": 0.0}
        try:
            agg[key]["value"] += int(h.get("value") or 0)
        except:
            pass
        try:
            if h.get("shares") is not None:
                agg[key]["shares"] += float(h.get("shares"))
        except:
            pass
    return agg


In [ ]:
def run_13f_summary(start_date=START_DATE, end_date=END_DATE, top_n=TOP_FILERS_N, min_change=MIN_CHANGE_USD, include_etfs=INCLUDE_ETFS, headers=None):
    headers = headers or HEADERS
    year = start_date.year
    # naive quarter selection for timeframe within single quarter; for multi-quarter expand this logic
    qtr = 1
    master_text = fetch_master_index(year, qtr, headers=headers)
    filings = parse_master_idx_for_13f(master_text, start_date, end_date)
    if not filings:
        logging.error("No 13F-HR filings found in the master index for the date range")
        return None
    filer_holdings = defaultdict(list)
    for f in filings:
        try:
            txt = fetch_filing_txt(f["filename"], headers=headers)
            xml_block, xml_filename = locate_information_table_xml_from_text(txt)
            xml_text = None
            if xml_block:
                xml_text = xml_block
            elif xml_filename:
                if xml_filename.startswith("http"):
                    xml_path = xml_filename
                else:
                    dirpath = "/".join(f["filename"].split("/")[:-1]) + "/"
                    xml_path = dirpath + xml_filename
                xml_text = fetch_xml_by_path(xml_path, headers=headers)
            else:
                logging.warning("Could not find XML informationTable for filing: %s %s", f["company_name"], f["filename"])
                continue
            holdings = parse_information_table_xml(xml_text)
            filer_key = f"{f['cik']}|{f['company_name']}"
            filer_holdings[filer_key].append({"date": f["date_filed"], "holdings": holdings})
            time.sleep(0.2)
        except Exception as e:
            logging.exception("Error processing filing %s: %s", f.get("filename"), e)
    filer_summaries = {}
    for filer, entries in filer_holdings.items():
        if not entries:
            continue
        entries_sorted = sorted(entries, key=lambda x: x["date"])
        earliest = entries_sorted[0]
        latest = entries_sorted[-1]
        earliest_agg = aggregate_holdings_by_security(earliest["holdings"])
        latest_agg = aggregate_holdings_by_security(latest["holdings"])
        total_assets = sum([v["value"] for v in latest_agg.values()])
        filer_summaries[filer] = {
            "earliest_date": earliest["date"],
            "latest_date": latest["date"],
            "earliest_agg": earliest_agg,
            "latest_agg": latest_agg,
            "total_assets": total_assets
        }
    if not filer_summaries:
        logging.error("No filer summaries produced; aborting")
        return None
    sorted_filers = sorted(filer_summaries.items(), key=lambda x: x[1]["total_assets"], reverse=True)
    top_filers = sorted_filers[:top_n]
    logging.info("Selected top %d filers by reported assets (within timeframe)", len(top_filers))
    aggregated = {}
    for filer, summary in top_filers:
        latest = summary["latest_agg"]
        earliest = summary["earliest_agg"]
        for key, lv in latest.items():
            if (not include_etfs) and is_etf(lv):
                continue
            en = earliest.get(key)
            ev = en["value"] if en is not None else 0
            net_change = lv["value"] - ev
            if key not in aggregated:
                aggregated[key] = {
                    "name": lv.get("name"),
                    "cusip": lv.get("cusip"),
                    "latest_value": 0,
                    "latest_shares": 0.0,
                    "net_change": 0
                }
            aggregated[key]["latest_value"] += lv.get("value", 0)
            try:
                aggregated[key]["latest_shares"] += lv.get("shares") or 0.0
            except:
                pass
            aggregated[key]["net_change"] += net_change
        for key, ev_item in earliest.items():
            if key in latest:
                continue
            if (not include_etfs) and is_etf(ev_item):
                continue
            if key not in aggregated:
                aggregated[key] = {
                    "name": ev_item.get("name"),
                    "cusip": ev_item.get("cusip"),
                    "latest_value": 0,
                    "latest_shares": 0.0,
                    "net_change": 0
                }
            aggregated[key]["net_change"] += (0 - ev_item.get("value", 0))
    rows = []
    for key, v in aggregated.items():
        rows.append({
            "key": key,
            "name": v.get("name"),
            "cusip": v.get("cusip"),
            "latest_value_usd": v.get("latest_value", 0),
            "latest_shares": v.get("latest_shares", 0.0),
            "net_change_usd": v.get("net_change", 0)
        })
    df = pd.DataFrame(rows)
    if df.empty:
        logging.error("Aggregated DataFrame is empty; no securities after filtering")
        return None
    df_buys = df[df["net_change_usd"] >= min_change].sort_values("net_change_usd", ascending=False)
    df_sells = df[df["net_change_usd"] <= -min_change].sort_values("net_change_usd")
    df_top_holdings = df.sort_values("latest_value_usd", ascending=False)
    top10_holdings = df_top_holdings.head(10).copy()
    top10_buys = df_buys.head(10).copy()
    top10_sells = df_sells.head(10).copy()
    # Save CSVs
    top10_holdings.to_csv(f"{OUTPUT_PREFIX}_top10_holdings.csv", index=False)
    top10_buys.to_csv(f"{OUTPUT_PREFIX}_top10_buys.csv", index=False)
    top10_sells.to_csv(f"{OUTPUT_PREFIX}_top10_sells.csv", index=False)
    df.to_csv(f"{OUTPUT_PREFIX}_all_aggregated.csv", index=False)
    # Return results for interactive display
    return {
        "top10_holdings": top10_holdings,
        "top10_buys": top10_buys,
        "top10_sells": top10_sells,
        "all_aggregated": df
    }


In [ ]:
# Run the analysis (uncomment to execute)
# WARNING: This will make requests to sec.gov. Ensure USER_AGENT above has a valid contact string.
# results = run_13f_summary()
# if results is not None:
#     print("Top 10 holdings:")
#     display(results['top10_holdings'])
#     print("Top 10 buys:")
#     display(results['top10_buys'])
#     print("Top 10 sells:")
#     display(results['top10_sells'])
#     print('CSV files saved with prefix:', OUTPUT_PREFIX)
print("Notebook ready. Edit USER_AGENT if needed and then run the final cell to fetch & analyze filings.")
